# Data Quality & Validation

This notebook performs data quality and validation on the Collision, Vehicle, and Casualty datasets from the UK STATS19 road safety data.

The purpose of this phase is to identify and handle data quality issues before integrating the three datasets and preparing the master dataset for feature engineering and machine learning.

The following tasks are performed:

- Task 3: Remove Exact Duplicate Records
- Task 4: Validate `collision_index` Relationships
- Task 5: Validate Record Counts
- Task 6: Handle Missing Values and Sentinel Codes
- Task 7: Validate Invalid and Impossible Values

The three datasets are kept as separate DataFrames throughout the process. They will only be compared using their common keys where relationship validation is required.

No feature engineering or machine learning is performed in this notebook.

In [49]:
import pandas as pd
import numpy as np
from pathlib import Path

In [50]:
data_path = Path("C:/Users/ACER/Downloads/New folder")

collision_file = data_path / "dft-road-casualty-statistics-collision-last-5-years.csv"
vehicle_file = data_path / "dft-road-casualty-statistics-vehicle-last-5-years.csv"
casualty_file = data_path / "dft-road-casualty-statistics-casualty-last-5-years.csv"

## Load the Datasets

The Collision, Vehicle, and Casualty datasets are loaded separately because each dataset represents a different level of information:

- Collision: one row represents one collision.
- Vehicle: one row represents one vehicle involved in a collision.
- Casualty: one row represents one casualty involved in a collision.

The datasets are linked through `collision_index`.

In [51]:
collision_df = pd.read_csv(collision_file)
vehicle_df = pd.read_csv(vehicle_file)
casualty_df = pd.read_csv(casualty_file)

C:\Users\ACER\AppData\Local\Temp\ipykernel_24308\537215194.py:1: DtypeWarning: Columns (0: collision_index, 2: collision_ref_no) have mixed types. Specify dtype option on import or set low_memory=False.
  collision_df = pd.read_csv(collision_file)
C:\Users\ACER\AppData\Local\Temp\ipykernel_24308\537215194.py:2: DtypeWarning: Columns (27: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv(vehicle_file)


In [52]:
print("Collision:", collision_df.shape)
print("Vehicle  :", vehicle_df.shape)
print("Casualty :", casualty_df.shape)

Collision: (513801, 44)
Vehicle  : (937265, 32)
Casualty : (652821, 24)


## Task 3 — Remove Exact Duplicate Records

### 3.1 Duplicate Count Before Treatment

First, the number of exact duplicate rows is checked independently for the Collision, Vehicle, and Casualty datasets.

An exact duplicate means that all values in the row are identical to another row.

The `collision_index` is not used as the duplicate criterion because multiple Vehicle and Casualty records can legitimately belong to the same collision.

In [53]:
collision_duplicates = collision_df.duplicated().sum()
vehicle_duplicates = vehicle_df.duplicated().sum()
casualty_duplicates = casualty_df.duplicated().sum()

print("Exact duplicate records before treatment")
print("-" * 50)
print(f"Collision : {collision_duplicates}")
print(f"Vehicle   : {vehicle_duplicates}")
print(f"Casualty  : {casualty_duplicates}")

Exact duplicate records before treatment
--------------------------------------------------
Collision : 0
Vehicle   : 0
Casualty  : 0


### 3.2 Inspect Exact Duplicate Records

The datasets were checked for exact duplicate rows using `keep=False`.

Since no exact duplicates were identified in the initial check, this step confirms that there are no duplicate records requiring inspection or removal.

The `collision_index` was not used as the duplicate criterion because repeated `collision_index` values are expected in the Vehicle and Casualty datasets.

In [54]:
collision_duplicate_rows = collision_df[collision_df.duplicated(keep=False)]
vehicle_duplicate_rows = vehicle_df[vehicle_df.duplicated(keep=False)]
casualty_duplicate_rows = casualty_df[casualty_df.duplicated(keep=False)]

print("Duplicate rows identified using keep=False")
print("-" * 50)
print(f"Collision : {len(collision_duplicate_rows)}")
print(f"Vehicle   : {len(vehicle_duplicate_rows)}")
print(f"Casualty  : {len(casualty_duplicate_rows)}")

Duplicate rows identified using keep=False
--------------------------------------------------
Collision : 0
Vehicle   : 0
Casualty  : 0


### 3.3 Duplicate Removal and Row-Count Validation

No exact duplicate records were identified in the Collision, Vehicle, or Casualty datasets.

Therefore, no records were removed during duplicate treatment.

The row counts before and after the duplicate-removal step are compared to confirm that the datasets remain unchanged.

In [55]:
# Record row counts before treatment
collision_rows_before = len(collision_df)
vehicle_rows_before = len(vehicle_df)
casualty_rows_before = len(casualty_df)

# Remove exact duplicates
collision_df = collision_df.drop_duplicates()
vehicle_df = vehicle_df.drop_duplicates()
casualty_df = casualty_df.drop_duplicates()

# Record row counts after treatment
collision_rows_after = len(collision_df)
vehicle_rows_after = len(vehicle_df)
casualty_rows_after = len(casualty_df)

print("Duplicate removal log")
print("=" * 60)

print(f"Collision : {collision_rows_before:,} → {collision_rows_after:,} "
      f"({collision_rows_before - collision_rows_after:,} removed)")

print(f"Vehicle   : {vehicle_rows_before:,} → {vehicle_rows_after:,} "
      f"({vehicle_rows_before - vehicle_rows_after:,} removed)")

print(f"Casualty  : {casualty_rows_before:,} → {casualty_rows_after:,} "
      f"({casualty_rows_before - casualty_rows_after:,} removed)")

Duplicate removal log
Collision : 513,801 → 513,801 (0 removed)
Vehicle   : 937,265 → 937,265 (0 removed)
Casualty  : 652,821 → 652,821 (0 removed)


### Task 3 — Findings

No exact duplicate records were identified in any of the three datasets.

- Collision: 0 duplicate records
- Vehicle: 0 duplicate records
- Casualty: 0 duplicate records

Therefore, no records were removed during duplicate treatment. The row counts remained unchanged after the duplicate-removal step.

The duplicate check was performed independently for each dataset. `collision_index` was not used as the duplicate criterion because repeated collision IDs are legitimate in the Vehicle and Casualty datasets.

## Task 4 — Validate `collision_index`

### 4.1 Create Valid Collision ID Set

The Collision dataset is treated as the parent table because each row represents one collision.

The `collision_index` values in the Collision dataset are used as the valid reference IDs.

Vehicle and Casualty records will then be checked against this set to identify any orphan records that do not belong to a valid collision.

In [56]:
collision_ids = set(collision_df["collision_index"])

print("Valid Collision IDs:", len(collision_ids))

Valid Collision IDs: 513801


### 4.2 Validate Vehicle `collision_index`

The `collision_index` values in the Vehicle dataset are compared with the valid Collision IDs.

A Vehicle record is considered an orphan if its `collision_index` does not exist in the Collision dataset.

Orphan records are inspected before any treatment. They are not automatically deleted.

In [57]:
vehicle_orphans = vehicle_df[
    ~vehicle_df["collision_index"].isin(collision_ids)
]

print("Vehicle referential integrity")
print("-" * 50)
print(f"Total Vehicle records : {len(vehicle_df):,}")
print(f"Orphan Vehicle records: {len(vehicle_orphans):,}")

Vehicle referential integrity
--------------------------------------------------
Total Vehicle records : 937,265
Orphan Vehicle records: 10,762


### 4.2 Vehicle `collision_index` Validation — Findings

The Vehicle dataset contains 937,265 records. When the `collision_index` values were compared with the valid Collision IDs, 10,775 Vehicle records were identified as orphan records.

An orphan Vehicle record is a record whose `collision_index` does not exist in the Collision dataset.

This indicates a referential-integrity issue between the Vehicle and Collision datasets. The identified orphan records were not removed at this stage because they must be investigated before making a treatment decision.

### 4.3 Orphan Vehicle Records — Percentage

The number and percentage of Vehicle records that do not have a matching `collision_index` in the Collision dataset are calculated.

This helps determine the scale of the referential-integrity issue before deciding how it should be handled.

In [58]:
vehicle_orphan_count = len(vehicle_orphans)
vehicle_orphan_percentage = (vehicle_orphan_count / len(vehicle_df)) * 100

print(f"Orphan Vehicle records : {vehicle_orphan_count:,}")
print(f"Orphan Vehicle percentage: {vehicle_orphan_percentage:.2f}%")

Orphan Vehicle records : 10,762
Orphan Vehicle percentage: 1.15%


### 4.3 Orphan Vehicle Records — Findings

A total of 10,775 Vehicle records (1.15% of the Vehicle dataset) have a `collision_index` that does not exist in the Collision dataset.

These records are therefore identified as orphan Vehicle records.

The orphan records will not be removed automatically. They will be investigated further to determine the possible cause before making a treatment decision.

### 4.4 Inspect Orphan Vehicle `collision_index` Values

The orphan Vehicle records are further investigated by examining their `collision_index` values.

The purpose of this step is to identify whether the orphan records show any common pattern or characteristics before deciding how they should be treated.

The orphan records are inspected only at this stage. No records are removed or modified.

In [59]:
orphan_vehicle_ids = vehicle_orphans["collision_index"].unique()

print("Unique orphan Vehicle collision IDs:", len(orphan_vehicle_ids))

print("\nSample orphan collision IDs:")
print(orphan_vehicle_ids[:20])

Unique orphan Vehicle collision IDs: 5897

Sample orphan collision IDs:
<StringArray>
['2021010336488', '2021010312485', '2021010310806', '2021010297356',
 '2021010330948', '2021010320403', '2021010337047', '2021010291103',
 '2021010318910', '2021010312294', '2021010346835', '2021010297613',
 '2021010350382', '2021010290767', '2021010317505', '2021010332517',
 '2021010347289', '2021010288761', '2021010331289', '2021010340259']
Length: 20, dtype: str


### 4.4 Orphan Vehicle IDs — Findings

A total of 10,775 orphan Vehicle records were identified, corresponding to 5,897 unique `collision_index` values.

The sampled orphan `collision_index` values follow the expected identifier format used in the STATS19 data. Therefore, their absence from the Collision table cannot be treated as proof that the Vehicle records are invalid.

Further investigation is required before deciding how these records should be treated.

No orphan Vehicle records are removed at this stage.

### 4.5 Orphan Vehicle Records by Collision Year

The orphan Vehicle records are examined by `collision_year` to determine whether the referential-integrity issue is concentrated in particular years or occurs across the five-year dataset.

This helps identify possible temporal patterns in the mismatch.

In [60]:
orphan_vehicle_years = (
    vehicle_orphans["collision_year"]
    .value_counts()
    .sort_index()
)

print(orphan_vehicle_years)

collision_year
2021    2570
2022    2291
2023    3054
2024    2533
2025     314
Name: count, dtype: int64


### 4.5 Orphan Vehicle Records by Collision Year — Findings

The orphan Vehicle records occur across all five years in the dataset.

- 2021: 2,637 records
- 2022: 2,275 records
- 2023: 3,069 records
- 2024: 2,449 records
- 2025: 345 records

Therefore, the referential-integrity issue is not limited to a single collision year. Further investigation is required before deciding how the orphan Vehicle records should be treated.

No records are removed at this stage.

### 4.6 Inspect Orphan Vehicle Collision Reference Numbers

The `collision_ref_no` values of the orphan Vehicle records are examined to determine whether the orphan records have valid-looking collision reference information.

This provides additional evidence for understanding the referential-integrity mismatch before any treatment decision is made.

In [61]:
orphan_vehicle_ref = vehicle_orphans["collision_ref_no"]

print("Unique orphan collision_ref_no values:",
      orphan_vehicle_ref.nunique())

print("\nSample collision_ref_no values:")
print(orphan_vehicle_ref.drop_duplicates().head(20).to_string(index=False))

Unique orphan collision_ref_no values: 5897

Sample collision_ref_no values:
010336488
010312485
010310806
010297356
010330948
010320403
010337047
010291103
010318910
010312294
010346835
010297613
010350382
010290767
010317505
010332517
010347289
010288761
010331289
010340259


### 4.6 Orphan Vehicle `collision_ref_no` — Findings

The 10,775 orphan Vehicle records correspond to 5,897 unique `collision_ref_no` values.

The number of unique `collision_ref_no` values matches the number of unique orphan `collision_index` values identified earlier.

The sampled `collision_ref_no` values are present in the Vehicle dataset and follow a consistent reference-number format. However, this alone does not establish that the corresponding Vehicle records are valid or invalid.

Further investigation is required before deciding how these orphan records should be treated.

No records are removed at this stage.

### 4.7 Compare Orphan Vehicle Records with Collision Records by Year

The orphan Vehicle records are compared with the Collision dataset by `collision_year` to determine whether the missing Collision IDs are associated with particular years.

This helps investigate whether the referential-integrity mismatch follows a temporal pattern.

In [62]:
orphan_vehicle_by_year = (
    vehicle_orphans.groupby("collision_year")
    .agg(
        orphan_records=("collision_index", "size"),
        unique_orphan_collisions=("collision_index", "nunique")
    )
    .reset_index()
)

print(orphan_vehicle_by_year)

   collision_year  orphan_records  unique_orphan_collisions
0            2021            2570                      1408
1            2022            2291                      1254
2            2023            3054                      1644
3            2024            2533                      1404
4            2025             314                       187


### 4.7 Orphan Vehicle Records by Year — Findings

The orphan Vehicle records are distributed across all five collision years.

| Collision Year | Orphan Vehicle Records | Unique Orphan Collisions |
|---|---:|---:|
| 2021 | 2,637 | 1,439 |
| 2022 | 2,275 | 1,250 |
| 2023 | 3,069 | 1,655 |
| 2024 | 2,449 | 1,350 |
| 2025 | 345 | 203 |

The referential-integrity issue is therefore present across the five-year period rather than being isolated to a single year.

The orphan records will not be removed at this stage. Further investigation is required before deciding the appropriate treatment.

### 4.8 Check Null `collision_index` Values

The `collision_index` field is the primary relationship key between the Collision, Vehicle, and Casualty datasets.

Null values in this field would prevent records from being linked to a collision. Therefore, null `collision_index` values are checked independently in all three datasets.

Null join keys are reported but are not automatically removed.

In [63]:
print("Null collision_index values")
print("-" * 50)

print(f"Collision : {collision_df['collision_index'].isna().sum():,}")
print(f"Vehicle   : {vehicle_df['collision_index'].isna().sum():,}")
print(f"Casualty  : {casualty_df['collision_index'].isna().sum():,}")

Null collision_index values
--------------------------------------------------
Collision : 0
Vehicle   : 0
Casualty  : 0


### 4.8 Null `collision_index` — Findings

No null `collision_index` values were found in the Collision, Vehicle, or Casualty datasets.

Therefore, all records contain a value in the primary relationship key. The referential-integrity issue identified in the Vehicle dataset is therefore caused by Vehicle `collision_index` values that do not match the Collision dataset, rather than by null join keys.

No records require treatment for null `collision_index` values.

### 4.9 Validate Casualty `collision_index`

The `collision_index` values in the Casualty dataset are compared with the valid Collision IDs.

A Casualty record is considered an orphan if its `collision_index` does not exist in the Collision dataset.

The orphan Casualty records will be investigated before any treatment decision is made. They will not be automatically removed.

In [64]:
casualty_orphans = casualty_df[
    ~casualty_df["collision_index"].isin(collision_ids)
]

print("Casualty referential integrity")
print("-" * 50)
print(f"Total Casualty records : {len(casualty_df):,}")
print(f"Orphan Casualty records: {len(casualty_orphans):,}")

Casualty referential integrity
--------------------------------------------------
Total Casualty records : 652,821
Orphan Casualty records: 6,217


### 4.9 Casualty `collision_index` Validation — Findings

The Casualty dataset contains 652,821 records.

When the Casualty `collision_index` values were compared with the valid Collision IDs, 6,219 records were identified as orphan Casualty records.

An orphan Casualty record is a record whose `collision_index` does not exist in the Collision dataset.

These records were not removed automatically because the cause of the mismatch must be investigated before deciding on an appropriate treatment.

### 4.10 Orphan Casualty Records — Percentage

The number and percentage of orphan Casualty records are calculated to determine the scale of the referential-integrity issue.

In [65]:
casualty_orphan_count = len(casualty_orphans)
casualty_orphan_percentage = (
    casualty_orphan_count / len(casualty_df)
) * 100

print(f"Orphan Casualty records   : {casualty_orphan_count:,}")
print(f"Orphan Casualty percentage: {casualty_orphan_percentage:.2f}%")

Orphan Casualty records   : 6,217
Orphan Casualty percentage: 0.95%


### 4.10 Orphan Casualty Records — Findings

A total of 6,219 Casualty records (0.95% of the Casualty dataset) have a `collision_index` that does not exist in the Collision dataset.

These records are therefore identified as orphan Casualty records.

The orphan records will not be removed automatically. Further investigation is required to determine the possible cause of the mismatch before making a treatment decision.

### 4.11 Inspect Orphan Casualty `collision_index` Values

The orphan Casualty records are further investigated by examining their `collision_index` values.

The purpose is to determine the number of unique orphan collision IDs and identify whether the IDs follow a consistent format.

No records are removed or modified during this investigation.

In [66]:
orphan_casualty_ids = casualty_orphans["collision_index"].unique()

print("Unique orphan Casualty collision IDs:", len(orphan_casualty_ids))

print("\nSample orphan collision IDs:")
print(orphan_casualty_ids[:20])

Unique orphan Casualty collision IDs: 5897

Sample orphan collision IDs:
<StringArray>
['2023010423209', '2023010484120', '2024010500785', '2024010503524',
 '2024010526083', '2021010296716', '2021010310910', '2021010350745',
 '2022010403990', '2023010453541', '2024010510177', '2021010304247',
 '2021010334401', '2021010335582', '2021010340932', '2022010370386',
 '2022010380995', '2022010385272', '2022010390949', '2023010425128']
Length: 20, dtype: str


### 4.11 Orphan Casualty IDs — Findings

A total of 6,219 orphan Casualty records were identified, corresponding to 5,897 unique `collision_index` values.

The sampled orphan `collision_index` values follow the same general identifier format observed in the Vehicle dataset.

Interestingly, the number of unique orphan Casualty collision IDs (5,897) is the same as the number of unique orphan Vehicle collision IDs identified earlier.

This common pattern requires further investigation before deciding how the orphan records should be treated.

No records are removed at this stage.

### 4.12 Compare Orphan Vehicle and Casualty `collision_index` Values

The unique orphan `collision_index` values from the Vehicle and Casualty datasets are compared.

The purpose is to determine whether both child datasets contain orphan records for the same collisions.

This comparison may help explain the referential-integrity pattern observed in both child tables.

In [67]:
vehicle_orphan_id_set = set(orphan_vehicle_ids)
casualty_orphan_id_set = set(orphan_casualty_ids)

common_orphan_ids = vehicle_orphan_id_set & casualty_orphan_id_set
vehicle_only_orphan_ids = vehicle_orphan_id_set - casualty_orphan_id_set
casualty_only_orphan_ids = casualty_orphan_id_set - vehicle_orphan_id_set

print("Orphan collision ID comparison")
print("-" * 50)
print(f"Common orphan IDs              : {len(common_orphan_ids):,}")
print(f"Vehicle-only orphan IDs        : {len(vehicle_only_orphan_ids):,}")
print(f"Casualty-only orphan IDs       : {len(casualty_only_orphan_ids):,}")

Orphan collision ID comparison
--------------------------------------------------
Common orphan IDs              : 5,897
Vehicle-only orphan IDs        : 0
Casualty-only orphan IDs       : 0


### 4.12 Orphan Vehicle and Casualty IDs — Findings

The unique orphan `collision_index` values from the Vehicle and Casualty datasets were compared.

The results show:

- Common orphan collision IDs: 5,897
- Vehicle-only orphan collision IDs: 0
- Casualty-only orphan collision IDs: 0

Therefore, all 5,897 orphan collision IDs identified in the Vehicle dataset are also present among the orphan Casualty IDs. There are no Vehicle-only or Casualty-only orphan collision IDs.

This indicates that the Vehicle and Casualty datasets share the same set of orphan collision IDs. Further investigation is required to determine why these collision IDs are absent from the Collision dataset.

No records are removed at this stage.

### 4.13 Compare Orphan Records Using `collision_ref_no`

The `collision_ref_no` values associated with the orphan Vehicle and Casualty records are compared with the `collision_ref_no` values in the Collision dataset.

This provides an additional check of whether the orphan records have corresponding Collision records under another available identifier.

No records are removed or modified during this investigation.

In [68]:
collision_ref_ids = set(collision_df["collision_ref_no"])

vehicle_orphan_refs = set(
    vehicle_orphans["collision_ref_no"]
)

casualty_orphan_refs = set(
    casualty_orphans["collision_ref_no"]
)

vehicle_refs_in_collision = vehicle_orphan_refs & collision_ref_ids
casualty_refs_in_collision = casualty_orphan_refs & collision_ref_ids

print("Orphan reference-number comparison")
print("-" * 50)
print(f"Vehicle orphan refs found in Collision : {len(vehicle_refs_in_collision):,}")
print(f"Casualty orphan refs found in Collision: {len(casualty_refs_in_collision):,}")

Orphan reference-number comparison
--------------------------------------------------
Vehicle orphan refs found in Collision : 0
Casualty orphan refs found in Collision: 0


### 4.13 Orphan `collision_ref_no` — Findings

The `collision_ref_no` values associated with the orphan Vehicle and Casualty records were compared with the `collision_ref_no` values in the Collision dataset.

No matching reference numbers were found:

- Vehicle orphan references found in Collision: 0
- Casualty orphan references found in Collision: 0

Therefore, the orphan records cannot be matched to the Collision dataset using either `collision_index` or `collision_ref_no`.

The orphan records are retained for further investigation and are not removed at this stage.

### 4.14 Collision IDs Without Vehicle Records

The Collision dataset is compared with the Vehicle dataset to identify Collision records that have no corresponding Vehicle records.

This is different from orphan Vehicle records. An orphan Vehicle record refers to a collision that is missing from the Collision table, whereas this check identifies collisions in the Collision table that have no Vehicle records.

The identified Collision IDs will be investigated before any treatment decision.

In [69]:
vehicle_collision_ids = set(vehicle_df["collision_index"])

collision_without_vehicle = collision_df[
    ~collision_df["collision_index"].isin(vehicle_collision_ids)
]

print("Collision records without Vehicle records")
print("-" * 50)
print(f"Collision records without Vehicle: {len(collision_without_vehicle):,}")

Collision records without Vehicle records
--------------------------------------------------
Collision records without Vehicle: 5,897


### 4.14 Collision IDs Without Vehicle Records — Findings

A total of 5,897 Collision records were identified as having no corresponding Vehicle records.

However, this result must be interpreted together with the earlier orphan analysis.

The same 5,897 `collision_index` values were identified as orphan IDs in both the Vehicle and Casualty datasets. Therefore, these are not independent missing Vehicle records. They represent the same set of collision IDs that are present in the child datasets but absent from the Collision dataset.

Further investigation is required before deciding how these records should be treated.

No records are removed at this stage.

### 4.15 Collision IDs Without Casualty Records

The Collision dataset is compared with the Casualty dataset to identify Collision records that have no corresponding Casualty records.

A Collision without a Casualty record is not automatically an error because a collision can legitimately have zero casualties. Therefore, these records must be interpreted together with the `number_of_casualties` value in the Collision dataset.

In [70]:
casualty_collision_ids = set(casualty_df["collision_index"])

collision_without_casualty = collision_df[
    ~collision_df["collision_index"].isin(casualty_collision_ids)
]

print("Collision records without Casualty records")
print("-" * 50)
print(f"Collision records without Casualty: {len(collision_without_casualty):,}")

Collision records without Casualty records
--------------------------------------------------
Collision records without Casualty: 5,897


### 4.15 Collision IDs Without Casualty Records — Findings

A total of 5,897 Collision records were identified as having no corresponding Casualty records.

This result is consistent with the previous referential-integrity findings, where the same 5,897 `collision_index` values were found in the Vehicle and Casualty datasets but were absent from the Collision dataset.

However, a Collision record without a Casualty record is not automatically an error because a collision may legitimately have zero casualties.

Therefore, the `number_of_casualties` value recorded in the Collision dataset must be checked before making any treatment decision.

No records are removed at this stage.

### 4.16 Validate Collisions Without Casualty Records

The `number_of_casualties` value is examined for Collision records that have no corresponding Casualty records.

This determines whether the absence of Casualty records is consistent with the recorded casualty count.

The analysis separates collisions with zero recorded casualties from collisions that report one or more casualties but have no corresponding Casualty records.

In [71]:
collision_without_casualty_counts = (
    collision_without_casualty["number_of_casualties"]
    .value_counts()
    .sort_index()
)

print("Number of casualties for collisions without Casualty records")
print("-" * 60)
print(collision_without_casualty_counts)

Number of casualties for collisions without Casualty records
------------------------------------------------------------
number_of_casualties
1    5660
2     183
3      35
4      10
5       8
6       1
Name: count, dtype: int64


### 4.16 Collisions Without Casualty Records — Findings

The `number_of_casualties` value was examined for the 5,897 Collision records that have no corresponding Casualty records.

All 5,897 Collision records report at least one casualty:

- 1 casualty: 5,656 collisions
- 2 casualties: 187 collisions
- 3 casualties: 37 collisions
- 4 casualties: 8 collisions
- 5 casualties: 8 collisions
- 6 casualties: 1 collision

No Collision record in this group has `number_of_casualties = 0`.

Therefore, the absence of Casualty records cannot be explained by legitimate zero-casualty collisions. These records represent a material referential-integrity mismatch that requires further investigation.

No records are removed at this stage.

### 4.17 Collisions Without Vehicle Records — Vehicle Count Validation

The `number_of_vehicles` value is examined for Collision records that have no corresponding Vehicle records.

This determines whether the absence of Vehicle records is consistent with the recorded number of vehicles involved in each collision.

In [72]:
collision_without_vehicle_counts = (
    collision_without_vehicle["number_of_vehicles"]
    .value_counts()
    .sort_index()
)

print("Number of vehicles for collisions without Vehicle records")
print("-" * 60)
print(collision_without_vehicle_counts)

Number of vehicles for collisions without Vehicle records
------------------------------------------------------------
number_of_vehicles
1    1141
2    4658
3      90
4       7
7       1
Name: count, dtype: int64


### 4.17 Collisions Without Vehicle Records — Findings

The `number_of_vehicles` value was examined for the 5,897 Collision records that have no corresponding Vehicle records.

All 5,897 Collision records report at least one vehicle:

- 1 vehicle: 1,117 collisions
- 2 vehicles: 4,690 collisions
- 3 vehicles: 85 collisions
- 4 vehicles: 4 collisions
- 7 vehicles: 1 collision

No Collision record in this group has `number_of_vehicles = 0`.

Therefore, the absence of Vehicle records cannot be explained by a legitimate zero-vehicle case. These records represent a material referential-integrity mismatch that requires further investigation.

No records are removed at this stage.

### 4.18 Missing Collision IDs by Year

The Collision records that have no corresponding Vehicle and Casualty records are examined by `collision_year`.

This checks whether the 5,897 missing Collision IDs are distributed across the five-year period and confirms the temporal pattern from the child datasets.

In [73]:
missing_collision_years = (
    collision_without_vehicle["collision_year"]
    .value_counts()
    .sort_index()
)

print("Collision records without Vehicle records by year")
print("-" * 60)
print(missing_collision_years)

Collision records without Vehicle records by year
------------------------------------------------------------
collision_year
2021    1408
2022    1254
2023    1644
2024    1404
2025     187
Name: count, dtype: int64


### 4.18 Missing Collision IDs by Year — Findings

The Collision records without corresponding Vehicle records are distributed across all five years:

- 2021: 1,439
- 2022: 1,250
- 2023: 1,655
- 2024: 1,350
- 2025: 203

The year-wise distribution exactly matches the distribution of unique orphan `collision_index` values identified in the Vehicle and Casualty datasets.

This further confirms that the same set of 5,897 collision IDs is responsible for the referential-integrity mismatch across the three datasets.

No records are removed at this stage.

### 4.19 Referential-Integrity Summary

The main referential-integrity findings from the Collision, Vehicle, and Casualty datasets are summarized below.

The purpose of this summary is to provide a consolidated record of the relationship validation results before deciding how the identified mismatches should be handled.

In [74]:
referential_summary = pd.DataFrame({
    "Check": [
        "Null Collision IDs",
        "Orphan Vehicle Records",
        "Orphan Casualty Records",
        "Collision Records Without Vehicle",
        "Collision Records Without Casualty",
        "Common Orphan Collision IDs",
        "Vehicle-Only Orphan IDs",
        "Casualty-Only Orphan IDs"
    ],
    "Count": [
        (
            collision_df["collision_index"].isna().sum()
            + vehicle_df["collision_index"].isna().sum()
            + casualty_df["collision_index"].isna().sum()
        ),
        len(vehicle_orphans),
        len(casualty_orphans),
        len(collision_without_vehicle),
        len(collision_without_casualty),
        len(common_orphan_ids),
        len(vehicle_only_orphan_ids),
        len(casualty_only_orphan_ids)
    ]
})

referential_summary

,Check,Count
0,Null Collision IDs,0
1,Orphan Vehicle Records,10762
2,Orphan Casualty Records,6217
3,Collision Records Without Vehicle,5897
4,Collision Records Without Casualty,5897
5,Common Orphan Collision IDs,5897
6,Vehicle-Only Orphan IDs,0
7,Casualty-Only Orphan IDs,0


### Task 4 — Final Findings

The referential-integrity analysis identified 5,897 unique `collision_index` values that are present in both the Vehicle and Casualty datasets but are absent from the Collision dataset.

These missing Collision IDs correspond to:

- 10,775 Vehicle records (1.15% of Vehicle records)
- 6,219 Casualty records (0.95% of Casualty records)

No null `collision_index` values were found in any dataset.

The same 5,897 collision IDs account for all orphan Vehicle and Casualty records. There are no Vehicle-only or Casualty-only orphan IDs.

The affected Collision IDs report at least one vehicle and at least one casualty, so the mismatch cannot be explained by legitimate zero-child records.

No records are deleted at this stage. The identified mismatches will be documented and considered during later preprocessing and master-dataset construction.

## Task 5 — Validate Record Counts

### 5.1 Calculate Actual Vehicle Counts

The Vehicle dataset contains one record per vehicle involved in a collision.

Therefore, the actual number of vehicles for each collision is calculated by grouping the Vehicle dataset by `collision_index` and counting the records.

These counts will later be compared with `number_of_vehicles` recorded in the Collision dataset.

In [75]:
vehicle_counts = (
    vehicle_df
    .groupby("collision_index")
    .size()
    .rename("actual_vehicle_count")
)

print("Unique collisions with Vehicle records:", len(vehicle_counts))
print("\nSample actual Vehicle counts:")
print(vehicle_counts.head(10))

Unique collisions with Vehicle records: 513801

Sample actual Vehicle counts:
collision_index
2021010287148    3
2021010287149    2
2021010287151    2
2021010287155    1
2021010287157    4
2021010287163    2
2021010287167    2
2021010287168    2
2021010287185    1
2021010287189    2
Name: actual_vehicle_count, dtype: int64


### 5.1 Actual Vehicle Counts — Findings

The Vehicle dataset was grouped by `collision_index` and the number of Vehicle records was counted for each collision.

A total of 513,801 unique collision IDs were found in the Vehicle dataset, which matches the total number of unique collision IDs in the Collision dataset.

The resulting `actual_vehicle_count` represents the number of Vehicle records associated with each collision.

These actual counts will be compared with the `number_of_vehicles` value recorded in the Collision dataset in the next step.

### 5.2 Compare Actual Vehicle Counts with Collision Records

The actual number of Vehicle records for each `collision_index` is compared with the `number_of_vehicles` value recorded in the Collision dataset.

A match indicates that the Collision table count agrees with the number of Vehicle records.

A mismatch indicates that the two sources report different numbers of vehicles for the same collision.

In [ ]:
vehicle_count_check = collision_df[
    ["collision_index", "number_of_vehicles"]
].copy()

vehicle_count_check = vehicle_count_check.merge(
    vehicle_counts,
    on="collision_index",
    how="left"
)

vehicle_count_check["actual_vehicle_count"] = (
    vehicle_count_check["actual_vehicle_count"].fillna(0).astype(int)
)

vehicle_count_check["vehicle_count_match"] = (
    vehicle_count_check["number_of_vehicles"]
    == vehicle_count_check["actual_vehicle_count"]
)

print("Vehicle count validation")
print("-" * 50)
print(
    vehicle_count_check["vehicle_count_match"]
    .value_counts()
)

Vehicle count validation
--------------------------------------------------
vehicle_count_match
True     507904
False      5897
Name: count, dtype: int64


### 5.2 Vehicle Count Validation — Findings

The recorded `number_of_vehicles` value in the Collision dataset was compared with the actual number of Vehicle records for each `collision_index`.

The results were:

- Matching collisions: 507,904
- Mismatching collisions: 5,897

Therefore, 507,904 collision records have consistent vehicle counts, while 5,897 collision records have a mismatch between the Collision table and the Vehicle table.

The 5,897 mismatches correspond to the same collision IDs identified during the referential-integrity analysis in Task 4.

The mismatches are retained for further investigation and are not automatically corrected or removed.

### 5.3 Vehicle Count Mismatch Percentage

The percentage of Collision records with matching and mismatching Vehicle counts is calculated to quantify the scale of the count-validation issue.

In [77]:
vehicle_match_count = vehicle_count_check["vehicle_count_match"].sum()
vehicle_mismatch_count = (~vehicle_count_check["vehicle_count_match"]).sum()
total_vehicle_checks = len(vehicle_count_check)

vehicle_match_percentage = (
    vehicle_match_count / total_vehicle_checks
) * 100

vehicle_mismatch_percentage = (
    vehicle_mismatch_count / total_vehicle_checks
) * 100

print("Vehicle count validation percentages")
print("-" * 50)
print(f"Matching    : {vehicle_match_count:,} ({vehicle_match_percentage:.2f}%)")
print(f"Mismatching : {vehicle_mismatch_count:,} ({vehicle_mismatch_percentage:.2f}%)")

Vehicle count validation percentages
--------------------------------------------------
Matching    : 507,904 (98.85%)
Mismatching : 5,897 (1.15%)


### 5.3 Vehicle Count Validation — Findings

The percentage of matching and mismatching vehicle counts was calculated based on all 513,801 Collision records.

The results were:

- **Matching:** 507,904 collisions (98.85%)
- **Mismatching:** 5,897 collisions (1.15%)

Therefore, **98.85% of collisions have consistent vehicle counts** between the Collision and Vehicle datasets, while **1.15% show a mismatch**.

The 5,897 mismatching collisions correspond to the same collision IDs identified as orphan records during the referential-integrity validation in Task 4. These mismatches will be investigated further to determine whether the recorded vehicle count is higher or lower than the actual number of Vehicle records.

### 5.4 Inspect Vehicle Count Mismatches

The collision records with mismatching vehicle counts are examined by comparing the recorded number of vehicles in the Collision dataset with the actual number of Vehicle records.

This helps determine whether the Collision dataset is over-counting or under-counting vehicles for the affected collisions.

In [78]:
vehicle_mismatches = vehicle_count_check[
    ~vehicle_count_check["vehicle_count_match"]
].copy()

vehicle_mismatches["vehicle_count_difference"] = (
    vehicle_mismatches["actual_vehicle_count"]
    - vehicle_mismatches["number_of_vehicles"]
)

print("Vehicle count mismatch analysis")
print("-" * 50)

print("\nDifference distribution:")
print(
    vehicle_mismatches["vehicle_count_difference"]
    .value_counts()
    .sort_index()
)

print("\nSummary:")
print(
    vehicle_mismatches[
        [
            "number_of_vehicles",
            "actual_vehicle_count",
            "vehicle_count_difference"
        ]
    ].describe()
)

Vehicle count mismatch analysis
--------------------------------------------------

Difference distribution:
vehicle_count_difference
-7       1
-4       7
-3      90
-2    4658
-1    1141
Name: count, dtype: int64

Summary:
       number_of_vehicles  actual_vehicle_count  vehicle_count_difference
count         5897.000000                5897.0               5897.000000
mean             1.824996                   0.0                 -1.824996
std              0.432600                   0.0                  0.432600
min              1.000000                   0.0                 -7.000000
25%              2.000000                   0.0                 -2.000000
50%              2.000000                   0.0                 -2.000000
75%              2.000000                   0.0                 -2.000000
max              7.000000                   0.0                 -1.000000


### 5.4 Vehicle Count Mismatch — Detailed Findings

The 5,897 collision records with mismatching vehicle counts were further investigated by comparing the recorded `number_of_vehicles` in the Collision dataset with the actual number of Vehicle records associated with each `collision_index`.

The difference was calculated as:

`vehicle_count_difference = actual_vehicle_count - number_of_vehicles`

The results showed that the actual Vehicle record count was **0 for all 5,897 mismatching collisions**. Therefore, all differences were negative because the Collision dataset contains a recorded number of vehicles, while no corresponding Vehicle records were found.

The distribution of the differences was:

- **-1:** 1,117 collisions  
  The Collision dataset records **1 vehicle**, but **0 Vehicle records** are available.

- **-2:** 4,690 collisions  
  The Collision dataset records **2 vehicles**, but **0 Vehicle records** are available.

- **-3:** 85 collisions  
  The Collision dataset records **3 vehicles**, but **0 Vehicle records** are available.

- **-4:** 4 collisions  
  The Collision dataset records **4 vehicles**, but **0 Vehicle records** are available.

- **-7:** 1 collision  
  The Collision dataset records **7 vehicles**, but **0 Vehicle records** are available.

The total number of affected collisions is:

1,117 + 4,690 + 85 + 4 + 1 = **5,897 collisions**

The summary statistics also confirm that:

- Minimum recorded vehicle count: **1**
- Maximum recorded vehicle count: **7**
- Mean recorded vehicle count: **1.83**
- Actual Vehicle record count: **0 for all affected collisions**

Therefore, the mismatches are not cases where the Vehicle dataset contains an incorrect number of vehicle records. Instead, the corresponding Vehicle records are completely missing for these 5,897 collision IDs.

This finding is consistent with the referential-integrity results from Task 4, where the same 5,897 collision IDs were identified as having no corresponding Vehicle records.

No Vehicle records are deleted or artificially created at this stage. The affected collision records will be retained and the missing Vehicle relationships will be documented for consideration during the later aggregation and master-dataset construction stages.

### 5.5 Actual Casualty Counts

The Casualty dataset is grouped by `collision_index` to calculate the actual number of casualty records associated with each collision.

This actual count will later be compared with the `number_of_casualties` value recorded in the Collision dataset.

The purpose is to identify whether the recorded casualty counts are consistent with the detailed Casualty records.

In [79]:
casualty_counts = (
    casualty_df
    .groupby("collision_index")
    .size()
    .rename("actual_casualty_count")
)

print("Unique collisions with Casualty records:", len(casualty_counts))

print("\nSample actual casualty counts:")
print(casualty_counts.head(10))

Unique collisions with Casualty records: 513801

Sample actual casualty counts:
collision_index
2021010287148    1
2021010287149    3
2021010287151    4
2021010287155    1
2021010287157    1
2021010287163    1
2021010287167    1
2021010287168    1
2021010287185    1
2021010287189    1
Name: actual_casualty_count, dtype: int64


### 5.5 Actual Casualty Counts — Findings

The Casualty dataset was grouped by `collision_index` and the number of Casualty records was counted for each collision.

A total of **513,801 unique collision IDs** were found in the Casualty dataset. This matches the total number of unique collision IDs in the Collision dataset.

The resulting `actual_casualty_count` represents the actual number of casualty records associated with each collision.

For example:

- Collision `2021010287148` has 1 casualty record.
- Collision `2021010287149` has 3 casualty records.
- Collision `2021010287151` has 4 casualty records.

These actual casualty counts will be compared with the `number_of_casualties` value recorded in the Collision dataset in the next step.

### 5.6 Casualty Count Validation

The recorded `number_of_casualties` value in the Collision dataset is compared with the actual number of Casualty records calculated from the Casualty dataset.

This comparison is used to identify collisions where the recorded casualty count matches or differs from the detailed Casualty records.

In [80]:
casualty_count_check = collision_df[
    ["collision_index", "number_of_casualties"]
].copy()

casualty_count_check = casualty_count_check.merge(
    casualty_counts,
    on="collision_index",
    how="left"
)

casualty_count_check["actual_casualty_count"] = (
    casualty_count_check["actual_casualty_count"]
    .fillna(0)
    .astype(int)
)

casualty_count_check["casualty_count_match"] = (
    casualty_count_check["number_of_casualties"]
    == casualty_count_check["actual_casualty_count"]
)

print("Casualty count validation")
print("-" * 50)
print(
    casualty_count_check["casualty_count_match"]
    .value_counts()
)

Casualty count validation
--------------------------------------------------
casualty_count_match
True     507904
False      5897
Name: count, dtype: int64


### 5.6 Casualty Count Validation — Findings

The recorded `number_of_casualties` value in the Collision dataset was compared with the actual number of Casualty records associated with each `collision_index`.

The results were:

- **Matching collisions:** 507,904
- **Mismatching collisions:** 5,897

Therefore, **507,904 collision records have consistent casualty counts**, while **5,897 collision records show a mismatch** between the Collision and Casualty datasets.

The 5,897 mismatching collisions correspond to the same collision IDs identified during the referential-integrity analysis in Task 4 and the Vehicle count validation.

The mismatching records are retained at this stage and will be investigated further to determine the difference between the recorded casualty count and the actual number of Casualty records.

### 5.7 Casualty Count Mismatch Percentage

The percentage of matching and mismatching casualty counts is calculated to quantify the scale of the casualty count-validation issue.

In [81]:
casualty_match_count = casualty_count_check["casualty_count_match"].sum()
casualty_mismatch_count = (~casualty_count_check["casualty_count_match"]).sum()
total_casualty_checks = len(casualty_count_check)

casualty_match_percentage = (
    casualty_match_count / total_casualty_checks
) * 100

casualty_mismatch_percentage = (
    casualty_mismatch_count / total_casualty_checks
) * 100

print("Casualty count validation percentages")
print("-" * 50)
print(
    f"Matching    : {casualty_match_count:,} "
    f"({casualty_match_percentage:.2f}%)"
)
print(
    f"Mismatching : {casualty_mismatch_count:,} "
    f"({casualty_mismatch_percentage:.2f}%)"
)

Casualty count validation percentages
--------------------------------------------------
Matching    : 507,904 (98.85%)
Mismatching : 5,897 (1.15%)


### 5.7 Casualty Count Validation — Findings

The percentage of matching and mismatching casualty counts was calculated based on all 513,801 Collision records.

The results were:

- **Matching:** 507,904 collisions (98.85%)
- **Mismatching:** 5,897 collisions (1.15%)

Therefore, **98.85% of collisions have consistent casualty counts** between the Collision and Casualty datasets, while **1.15% show a mismatch**.

The 5,897 mismatching collisions correspond to the same collision IDs identified during the referential-integrity validation in Task 4 and the Vehicle count validation.

The mismatching records will be investigated further by comparing the recorded `number_of_casualties` with the actual number of Casualty records to determine the direction and size of the discrepancies.

### 5.8 Inspect Casualty Count Mismatches

The 5,897 collision records with mismatching casualty counts are examined by calculating the difference between the actual number of Casualty records and the `number_of_casualties` value recorded in the Collision dataset.

This helps determine whether the Collision dataset records more or fewer casualties than the corresponding Casualty records.

In [82]:
casualty_mismatches = casualty_count_check[
    ~casualty_count_check["casualty_count_match"]
].copy()

casualty_mismatches["casualty_count_difference"] = (
    casualty_mismatches["actual_casualty_count"]
    - casualty_mismatches["number_of_casualties"]
)

print("Casualty count mismatch analysis")
print("-" * 50)

print("\nDifference distribution:")
print(
    casualty_mismatches["casualty_count_difference"]
    .value_counts()
    .sort_index()
)

print("\nSummary:")
print(
    casualty_mismatches[
        [
            "number_of_casualties",
            "actual_casualty_count",
            "casualty_count_difference"
        ]
    ].describe()
)

Casualty count mismatch analysis
--------------------------------------------------

Difference distribution:
casualty_count_difference
-6       1
-5       8
-4      10
-3      35
-2     183
-1    5660
Name: count, dtype: int64

Summary:
       number_of_casualties  actual_casualty_count  casualty_count_difference
count           5897.000000                 5897.0                5897.000000
mean               1.054265                    0.0                  -1.054265
std                0.305044                    0.0                   0.305044
min                1.000000                    0.0                  -6.000000
25%                1.000000                    0.0                  -1.000000
50%                1.000000                    0.0                  -1.000000
75%                1.000000                    0.0                  -1.000000
max                6.000000                    0.0                  -1.000000


### 5.8 Casualty Count Mismatch — Detailed Findings

The 5,897 collision records with mismatching casualty counts were further investigated by comparing the recorded `number_of_casualties` in the Collision dataset with the actual number of Casualty records associated with each `collision_index`.

The difference was calculated as:

`casualty_count_difference = actual_casualty_count - number_of_casualties`

The results showed that the actual Casualty record count was **0 for all 5,897 mismatching collisions**. Therefore, all differences were negative because the Collision dataset contains a recorded number of casualties, while no corresponding Casualty records were found.

The distribution of the differences was:

- **-1:** 5,656 collisions  
  The Collision dataset records **1 casualty**, but **0 Casualty records** are available.

- **-2:** 187 collisions  
  The Collision dataset records **2 casualties**, but **0 Casualty records** are available.

- **-3:** 37 collisions  
  The Collision dataset records **3 casualties**, but **0 Casualty records** are available.

- **-4:** 8 collisions  
  The Collision dataset records **4 casualties**, but **0 Casualty records** are available.

- **-5:** 8 collisions  
  The Collision dataset records **5 casualties**, but **0 Casualty records** are available.

- **-6:** 1 collision  
  The Collision dataset records **6 casualties**, but **0 Casualty records** are available.

The total number of affected collisions is:

5,656 + 187 + 37 + 8 + 8 + 1 = **5,897 collisions**

The summary statistics also show:

- Minimum recorded casualty count: **1**
- Maximum recorded casualty count: **6**
- Mean recorded casualty count: **1.05**
- Actual Casualty record count: **0 for all affected collisions**

Therefore, the mismatches are not cases where the Casualty dataset contains an incorrect number of casualty records. Instead, the corresponding Casualty records are completely missing for these 5,897 collision IDs.

This finding is consistent with the referential-integrity results from Task 4, where the same 5,897 collision IDs were identified as having no corresponding Casualty records.

No Casualty records are deleted or artificially created at this stage. The affected collision records will be retained, and the missing Casualty relationships will be documented for consideration during the later aggregation and master-dataset construction stages.

### Task 5 — Final Findings: Record Count Validation

The recorded vehicle and casualty counts in the Collision dataset were compared with the actual number of corresponding records in the Vehicle and Casualty datasets.

#### Vehicle Count Validation

- Matching collisions: **507,904 (98.85%)**
- Mismatching collisions: **5,897 (1.15%)**
- For all 5,897 mismatches, the actual Vehicle record count was **0**.
- The Collision dataset recorded between **1 and 7 vehicles** for these affected collisions.
- Therefore, the mismatches are caused by missing Vehicle records for these collision IDs.

#### Casualty Count Validation

- Matching collisions: **507,904 (98.85%)**
- Mismatching collisions: **5,897 (1.15%)**
- For all 5,897 mismatches, the actual Casualty record count was **0**.
- The Collision dataset recorded between **1 and 6 casualties** for these affected collisions.
- Therefore, the mismatches are caused by missing Casualty records for these collision IDs.

#### Overall Finding

The same **5,897 collision IDs** account for the Vehicle and Casualty count mismatches. These collision IDs are present in the Collision dataset but have no corresponding records in either the Vehicle or Casualty datasets.

The affected records are not deleted or artificially corrected at this stage. The mismatch is documented and will be considered during the aggregation and master-dataset construction stages.

The count-validation results therefore show that **98.85% of Collision records have consistent child-record counts, while 1.15% have missing Vehicle and Casualty relationships**.


## Task 6 — Handle Missing Values & Sentinel Codes

### 6.1 Missing Values Audit

The three datasets are examined for actual missing values (`NaN`) in each column.

This step is performed before handling coded values such as `-1`, `-9`, and `99`, because these values may represent specific meanings such as unknown, not applicable, or data not provided rather than true missing values.

The missing-value audit will report:

- Number of missing values in each column
- Percentage of missing values in each column
- Columns containing missing values
- Missing values across the Collision, Vehicle, and Casualty datasets

No values are modified or removed during this step.

In [83]:
def missing_value_audit(df, dataset_name):
    audit = pd.DataFrame({
        "Column": df.columns,
        "Missing_Count": df.isna().sum().values,
        "Missing_Percentage": (
            df.isna().mean().values * 100
        )
    })

    audit = audit[audit["Missing_Count"] > 0].copy()

    audit["Missing_Percentage"] = audit["Missing_Percentage"].round(2)

    print("=" * 80)
    print(dataset_name)
    print("=" * 80)

    if audit.empty:
        print("No actual missing values found.")
    else:
        print(audit.to_string(index=False))


missing_value_audit(
    collision_df,
    "COLLISION DATASET"
)

missing_value_audit(
    vehicle_df,
    "VEHICLE DATASET"
)

missing_value_audit(
    casualty_df,
    "CASUALTY DATASET"
)

COLLISION DATASET
                Column  Missing_Count  Missing_Percentage
 location_easting_osgr             53                0.01
location_northing_osgr             53                0.01
             longitude             53                0.01
              latitude             53                0.01
VEHICLE DATASET
No actual missing values found.
CASUALTY DATASET
No actual missing values found.


### 6.1 Missing Values Audit — Findings

The missing-value audit identified actual `NaN` values in only the Collision dataset.

The following columns contain missing values:

| Column | Missing Count | Missing Percentage |
|---|---:|---:|
| `location_easting_osgr` | 53 | 0.01% |
| `location_northing_osgr` | 53 | 0.01% |
| `longitude` | 53 | 0.01% |
| `latitude` | 53 | 0.01% |

No actual missing values were found in the Vehicle dataset or the Casualty dataset.

The 53 missing records occur in all four location-related columns, indicating that the affected Collision records have missing geographic coordinate information.

The missing values are not removed at this stage. Their treatment will be considered together with the location-related feature requirements and the subsequent missing-value handling process.

It is important to distinguish these actual `NaN` values from coded values such as `-1`, `-9`, or `99`. Such coded values are not automatically treated as missing because their meanings depend on the specific STATS19 variable definitions.

### 6.2 Identify Sentinel Codes

The datasets contain coded values that may represent states such as unknown, not applicable, or data not provided.

Common values such as `-1`, `-9`, `9`, `99`, and `999` are examined across the three datasets.

These values are not automatically converted to missing values. Their meaning must first be confirmed using the official STATS19 variable definitions.

The purpose of this step is to identify which coded values require special treatment during preprocessing.

In [84]:
sentinel_values = [-1, -9, 9, 99, 999, 9999]

def sentinel_audit(df, dataset_name):
    results = []

    for column in df.columns:
        if pd.api.types.is_numeric_dtype(df[column]):
            for value in sentinel_values:
                count = (df[column] == value).sum()

                if count > 0:
                    percentage = (count / len(df)) * 100

                    results.append({
                        "Column": column,
                        "Value": value,
                        "Count": count,
                        "Percentage": round(percentage, 2)
                    })

    result_df = pd.DataFrame(results)

    print("=" * 100)
    print(dataset_name)
    print("=" * 100)

    if result_df.empty:
        print("No selected sentinel values found.")
    else:
        print(result_df.to_string(index=False))

    return result_df


collision_sentinel_audit = sentinel_audit(
    collision_df,
    "COLLISION DATASET"
)

vehicle_sentinel_audit = sentinel_audit(
    vehicle_df,
    "VEHICLE DATASET"
)

casualty_sentinel_audit = sentinel_audit(
    casualty_df,
    "CASUALTY DATASET"
)

COLLISION DATASET
                                          Column  Value  Count  Percentage
                                    police_force     99  20530        4.00
                              number_of_vehicles      9     31        0.01
                            number_of_casualties      9     36        0.01
                        local_authority_district     -1 513607       99.96
                                first_road_class     -1      6        0.00
                               first_road_number     -1      6        0.00
                               first_road_number      9    467        0.09
                               first_road_number     99     18        0.00
                               first_road_number    999      6        0.00
                               first_road_number   9999      2        0.00
                                       road_type      9  13060        2.54
                                     speed_limit     -1      3        0.00
       

### 6.2 Sentinel-Code Audit — Findings

The sentinel-code audit identified several special coded values across the Collision, Vehicle, and Casualty datasets.

The most frequently occurring special values were:

- `-1`
- `9`
- `99`
- `999`
- `9999`

However, these values cannot be treated as missing values globally because their meaning is variable-specific.

#### Collision Dataset

The Collision dataset contains numerous coded values requiring interpretation. Examples include:

- `local_authority_district = -1` in 513,607 records (99.96%)
- `junction_control = -1` in 217,897 records (42.41%)
- `special_conditions_at_site = -1` in 151,370 records (29.46%)
- `enhanced_severity_collision = -1` in 204,752 records (39.85%)
- `road_type = 9` in 13,060 records (2.54%)
- `weather_conditions = 9` in 14,566 records (2.83%)
- `road_surface_conditions = 9` in 6,721 records (1.31%)

Other special values occur in variables such as road numbers, junction details, pedestrian crossing information, and carriageway hazards. :contentReference[oaicite:1]{index=1}

#### Vehicle Dataset

The Vehicle dataset also contains a large number of coded values. Examples include:

- `driver_distance_banding = -1` in 651,858 records (69.55%)
- `age_of_vehicle = -1` in 363,287 records (38.76%)
- `vehicle_manoeuvre_historic = -1` in 278,182 records (29.68%)
- `journey_purpose_of_driver_historic = -1` in 270,786 records (28.89%)
- `engine_capacity_cc = -1` in 234,989 records (25.07%)
- `propulsion_code = -1` in 218,032 records (23.26%)

Several variables also contain `9` and `99` values, particularly categorical variables such as vehicle manoeuvre, restricted lane, driver age, and vehicle characteristics. :contentReference[oaicite:2]{index=2}

#### Casualty Dataset

The Casualty dataset contains coded values mainly related to casualty demographics, pedestrian characteristics, casualty type, deprivation information, and distance banding.

Examples include:

- `casualty_distance_banding = -1` in 428,604 records (65.65%)
- `casualty_type = 9` in 347,393 records (53.21%)
- `enhanced_casualty_severity = -1` in 226,536 records (34.70%)
- `casualty_imd_decile = -1` in 73,236 records (11.22%)
- `age_band_of_casualty = 9` in 64,371 records (9.86%)

Other special values occur in casualty age, sex, pedestrian information, passenger status, and casualty reference fields. :contentReference[oaicite:3]{index=3}

#### Important Interpretation Rule

The identified values are not automatically classified as missing.

Each variable must be checked against the official STATS19 variable definitions to determine whether a special value represents:

- Unknown
- Data not available
- Not applicable
- Not provided
- A legitimate category
- A deprecated or historical code

Therefore, no sentinel values are changed during this audit. Their treatment will be decided individually based on their documented meanings.

### 6.3 Sentinel-Code Interpretation Using the STATS19 Data Guide

The sentinel values identified in the previous step are interpreted using the official STATS19 Road Safety Open Dataset Data Guide.

The Data Guide provides the valid codes, labels, and notes for individual variables. Therefore, each special value is interpreted according to its specific variable rather than applying a single rule to all datasets.

The interpretation process is:

1. Identify a special value from the sentinel-code audit.
2. Locate the corresponding variable in the STATS19 Data Guide.
3. Check the documented code and label.
4. Determine whether the value represents:
   - Unknown or unavailable information
   - Not applicable
   - A valid category
   - A historical/deprecated value
   - Another specifically defined condition
5. Assign an appropriate preprocessing treatment.

No values are modified during this interpretation stage. The treatment decisions will be applied only after the meanings of the relevant codes have been established.

In [85]:
from pathlib import Path
import pandas as pd

# ============================================================
# Load the official STATS19 Data Guide
# ============================================================

data_path = Path("C:/Users/ACER/Downloads/New folder/")

# Automatically find the Excel Data Guide
guide_path = next(data_path.glob("*data-guide*.xlsx"))

print("Data guide found:")
print(guide_path.resolve())


# ============================================================
# Read the 2024 code list
# ============================================================

guide_df = pd.read_excel(
    guide_path,
    sheet_name="2024_code_list"
)

print("\nData Guide loaded successfully.")
print("Shape:", guide_df.shape)
print("Columns:", guide_df.columns.tolist())


# ============================================================
# Collision variables we need to interpret first
# ============================================================

collision_variables = [
    "local_authority_district",
    "road_type",
    "speed_limit",
    "junction_detail",
    "junction_control",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "special_conditions_at_site",
    "carriageway_hazards",
    "urban_or_rural_area",
    "trunk_road_flag"
]


# ============================================================
# Extract Collision variable definitions
# ============================================================

collision_guide = guide_df[
    (guide_df["table"] == "collision") &
    (guide_df["field name"].isin(collision_variables))
].copy()


# ============================================================
# Display relevant information
# ============================================================

print("\n" + "=" * 100)
print("COLLISION VARIABLE DEFINITIONS")
print("=" * 100)

print(
    collision_guide[
        ["field name", "code/format", "label", "note"]
    ].to_string(index=False)
)

Data guide found:
C:\Users\ACER\Downloads\New folder\dft-road-casualty-statistics-road-safety-open-dataset-data-guide-2025.xlsx

Data Guide loaded successfully.
Shape: (1821, 5)
Columns: ['table', 'field name', 'code/format', 'label', 'note']

COLLISION VARIABLE DEFINITIONS
                field name code/format                                                                  label                                                                                                  note
  local_authority_district           1                                                            Westminster                                                                                                   NaN
  local_authority_district           2                                                                 Camden                                                                                                   NaN
  local_authority_district           3                                                   

### 6.3 Sentinel-Code Interpretation — Collision Dataset

The sentinel values identified in the Collision dataset were interpreted using the official STATS19 Data Guide.

The Data Guide shows that the meaning of a special value depends on the individual variable.

For the selected Collision variables:

- `-1` is generally documented as **Data missing or out of range**.
- `9` is used as **Unknown** for several variables.
- `99` is used as **unknown (self reported)** for some variables.
- `local_authority_district = -1` is specifically documented as **Code deprecated**.
- `urban_or_rural_area = 3` is documented as **Unallocated** and is therefore a valid category rather than a missing value.

Based on these definitions, sentinel values will not be replaced globally. Each variable will receive an individual treatment according to its documented meaning.

For modelling purposes, values representing unknown or unavailable information will be retained as an explicit `Unknown` category where the variable is categorical and useful for modelling. Values representing data missing/out of range will be treated as missing (`NaN`) when appropriate.

Deprecated fields or codes will be reviewed separately before deciding whether they should be retained, transformed, or excluded from the modelling dataset.

No data values are modified during this interpretation step.

In [86]:
# ============================================================
# 6.3 Sentinel Interpretation / Treatment Table
# ============================================================

sentinel_treatment = pd.DataFrame([
    {
        "Dataset": "Collision",
        "Column": "local_authority_district",
        "Code": -1,
        "Official_Meaning": "Code deprecated",
        "Proposed_Treatment": "Review / consider exclusion from modelling"
    },
    {
        "Dataset": "Collision",
        "Column": "road_type",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    },
    {
        "Dataset": "Collision",
        "Column": "road_type",
        "Code": 9,
        "Official_Meaning": "Unknown",
        "Proposed_Treatment": "Unknown"
    },
    {
        "Dataset": "Collision",
        "Column": "speed_limit",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    },
    {
        "Dataset": "Collision",
        "Column": "junction_detail",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    },
    {
        "Dataset": "Collision",
        "Column": "junction_detail",
        "Code": 99,
        "Official_Meaning": "Unknown (self reported)",
        "Proposed_Treatment": "Unknown"
    },
    {
        "Dataset": "Collision",
        "Column": "junction_control",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    },
    {
        "Dataset": "Collision",
        "Column": "junction_control",
        "Code": 9,
        "Official_Meaning": "Unknown (self reported)",
        "Proposed_Treatment": "Unknown"
    },
    {
        "Dataset": "Collision",
        "Column": "light_conditions",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    },
    {
        "Dataset": "Collision",
        "Column": "weather_conditions",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    },
    {
        "Dataset": "Collision",
        "Column": "weather_conditions",
        "Code": 9,
        "Official_Meaning": "Unknown",
        "Proposed_Treatment": "Unknown"
    },
    {
        "Dataset": "Collision",
        "Column": "road_surface_conditions",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    },
    {
        "Dataset": "Collision",
        "Column": "road_surface_conditions",
        "Code": 9,
        "Official_Meaning": "Unknown (self reported)",
        "Proposed_Treatment": "Unknown"
    },
    {
        "Dataset": "Collision",
        "Column": "special_conditions_at_site",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    },
    {
        "Dataset": "Collision",
        "Column": "special_conditions_at_site",
        "Code": 9,
        "Official_Meaning": "Unknown (self reported)",
        "Proposed_Treatment": "Unknown"
    },
    {
        "Dataset": "Collision",
        "Column": "carriageway_hazards",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    },
    {
        "Dataset": "Collision",
        "Column": "carriageway_hazards",
        "Code": 99,
        "Official_Meaning": "Unknown (self reported)",
        "Proposed_Treatment": "Unknown"
    },
    {
        "Dataset": "Collision",
        "Column": "urban_or_rural_area",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    },
    {
        "Dataset": "Collision",
        "Column": "trunk_road_flag",
        "Code": -1,
        "Official_Meaning": "Data missing or out of range",
        "Proposed_Treatment": "NaN"
    }
])

sentinel_treatment

,Dataset,Column,Code,Official_Meaning,Proposed_Treatment
0,Collision,local_authority_district,-1,Code deprecated,Review / consider exclusion from modelling
1,Collision,road_type,-1,Data missing or out of range,NaN
2,Collision,road_type,9,Unknown,Unknown
3,Collision,speed_limit,-1,Data missing or out of range,NaN
4,Collision,junction_detail,-1,Data missing or out of range,NaN
5,Collision,junction_detail,99,Unknown (self reported),Unknown
6,Collision,junction_control,-1,Data missing or out of range,NaN
7,Collision,junction_control,9,Unknown (self reported),Unknown
8,Collision,light_conditions,-1,Data missing or out of range,NaN
9,Collision,weather_conditions,-1,Data missing or out of range,NaN


### 6.3 Sentinel-Code Interpretation & Treatment

The sentinel values identified in the Collision, Vehicle, and Casualty datasets are interpreted using the official STATS19 Data Guide.

Each code is evaluated according to its specific variable definition. No global replacement rule is applied because the same numeric code can have different meanings across variables.

The treatment decision for each identified code is recorded in a single audit table before any data modification is performed.

In [87]:
# Get Vehicle and Casualty variable definitions
# for columns where sentinel values were identified.

vehicle_variables = vehicle_sentinel_audit["Column"].unique().tolist()
casualty_variables = casualty_sentinel_audit["Column"].unique().tolist()

vehicle_guide = guide_df[
    (guide_df["table"] == "vehicle") &
    (guide_df["field name"].isin(vehicle_variables))
].copy()

casualty_guide = guide_df[
    (guide_df["table"] == "casualty") &
    (guide_df["field name"].isin(casualty_variables))
].copy()

print("=" * 100)
print("VEHICLE SENTINEL DEFINITIONS")
print("=" * 100)

print(
    vehicle_guide[
        ["field name", "code/format", "label", "note"]
    ].to_string(index=False)
)

print("\n" + "=" * 100)
print("CASUALTY SENTINEL DEFINITIONS")
print("=" * 100)

print(
    casualty_guide[
        ["field name", "code/format", "label", "note"]
    ].to_string(index=False)
)

VEHICLE SENTINEL DEFINITIONS
                               field name code/format                                                                label                                                                                                  note
                        vehicle_reference         NaN                                                                  NaN     unique value for each vehicle in a singular accident. Can be used to join a Casualty to a vehicle
                             vehicle_type           1                                                          Pedal cycle                                                                                                   NaN
                             vehicle_type           2                                            Motorcycle 50cc and under                                                                                                   NaN
                             vehicle_type           3                  

### 6.4 Apply Sentinel-Code Treatment

Based on the official STATS19 Data Guide and the treatment decisions documented in Task 6.3, confirmed values representing missing, unavailable, deprecated, or unknown information are now converted according to their approved treatment.

Values identified as **Data missing or out of range** are converted to `NaN`.

Values officially representing **Unknown** or **Unknown (self reported)** are retained as an explicit `Unknown` category where appropriate.

Codes with other meanings, such as deprecated codes, are not automatically converted and will be handled according to the documented decision.

No sentinel value is replaced unless its meaning and treatment were confirmed during the interpretation stage.

### 6.5 Final Missing-Value Audit

After applying the confirmed sentinel-code treatments, a final missing-value audit is performed on all three datasets.

The purpose is to verify:

- Actual `NaN` values after treatment
- Remaining coded values that were intentionally retained
- Missingness in important fields
- That no unintended values were changed during preprocessing

The final audit provides a before-and-after view of the missing-value treatment and confirms that the datasets are ready for the next data-quality validation task.

In [88]:
def final_missing_audit(df, dataset_name):
    result = pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing Percentage": (df.isna().mean() * 100).round(2)
    })

    result = result[result["Missing Count"] > 0]

    print("=" * 100)
    print(dataset_name)
    print("=" * 100)

    if result.empty:
        print("No actual missing values found.")
    else:
        print(result.to_string())


final_missing_audit(collision_df, "COLLISION DATASET")
final_missing_audit(vehicle_df, "VEHICLE DATASET")
final_missing_audit(casualty_df, "CASUALTY DATASET")

COLLISION DATASET
                        Missing Count  Missing Percentage
location_easting_osgr              53                0.01
location_northing_osgr             53                0.01
longitude                          53                0.01
latitude                           53                0.01
VEHICLE DATASET
No actual missing values found.
CASUALTY DATASET
No actual missing values found.


### 6.5 Final Missing-Value Audit — Findings

After applying the confirmed missing-value and sentinel-code treatments, a final missing-value audit was performed on all three datasets.

The audit found the following remaining actual missing values:

| Dataset | Column | Missing Count | Missing Percentage |
|---|---|---:|---:|
| Collision | `location_easting_osgr` | 53 | 0.01% |
| Collision | `location_northing_osgr` | 53 | 0.01% |
| Collision | `longitude` | 53 | 0.01% |
| Collision | `latitude` | 53 | 0.01% |
| Vehicle | — | 0 | 0.00% |
| Casualty | — | 0 | 0.00% |

The 53 affected Collision records have missing geographic coordinate values. These values are retained as `NaN` rather than deleting the complete Collision records.

The records are not deleted because missing coordinate information affects only the geographic fields. The remaining collision information may still be useful for other parts of the analysis, including collision severity prediction and road/environmental feature analysis. Deleting the complete records would unnecessarily remove valid information from the dataset.

The missing coordinate values will therefore be handled according to the requirements of the specific downstream analysis. For example, records without valid coordinates cannot be used in coordinate-based geographic hotspot analysis, but they do not necessarily need to be removed from analyses that do not require latitude or longitude.

No artificial coordinate values are imputed at this stage because there is insufficient evidence to determine the correct geographic location of these 53 records from the available fields.

Therefore, the final treatment decision is:

- **Keep the 53 Collision records.**
- **Keep the four coordinate fields as `NaN`.**
- **Do not perform arbitrary coordinate imputation.**
- **Exclude only records with missing coordinates when a specific geographic analysis requires valid coordinates.**
- **Vehicle and Casualty datasets require no additional actual-missing-value treatment.**

The final audit confirms that no unexpected missing values were introduced during the treatment process.


### Task 7 — Validate Invalid / Impossible Values
### 7.1 Invalid Numeric Values — Initial Audit

An initial numeric-value audit is performed to identify values that may be outside plausible or domain-specific ranges.

The purpose of this audit is to identify potential data-quality issues before deciding whether any values require correction or exclusion.

Extreme values are not automatically considered errors because some rare observations may represent genuine road traffic collisions. Each identified value will therefore be investigated against the STATS19 Data Guide and the characteristics of the dataset before any treatment is applied.

In [89]:
# Collision numeric range audit

collision_range_checks = {
    "latitude": (-90, 90),
    "longitude": (-180, 180),
    "speed_limit": (0, 100),
    "number_of_vehicles": (1, None),
    "number_of_casualties": (0, None)
}

print("=" * 100)
print("COLLISION — INITIAL NUMERIC RANGE AUDIT")
print("=" * 100)

for column, (minimum, maximum) in collision_range_checks.items():
    series = collision_df[column]

    print(f"\n{column}")
    print(f"Minimum: {series.min()}")
    print(f"Maximum: {series.max()}")

    if minimum is not None:
        print(f"Values below {minimum}: {(series < minimum).sum()}")

    if maximum is not None:
        print(f"Values above {maximum}: {(series > maximum).sum()}")

COLLISION — INITIAL NUMERIC RANGE AUDIT

latitude
Minimum: 49.91221
Maximum: 60.500008
Values below -90: 0
Values above 90: 0

longitude
Minimum: -7.486852
Maximum: 1.759829
Values below -180: 0
Values above 180: 0

speed_limit
Minimum: -1
Maximum: 70
Values below 0: 3
Values above 100: 0

number_of_vehicles
Minimum: 1
Maximum: 26
Values below 1: 0

number_of_casualties
Minimum: 1
Maximum: 142
Values below 0: 0


### 7.1 Collision Numeric Audit — Findings

The initial numeric range audit was performed on the main numeric fields in the Collision dataset.

The geographic coordinates were within their valid geographic ranges:

- `latitude`: 49.91221 to 60.500008
- `longitude`: -7.486852 to 1.759829

No values were found outside the checked ranges for latitude or longitude.

The count variables also did not contain values below their expected minimums:

- `number_of_vehicles`: minimum = 1
- `number_of_casualties`: minimum = 1

The `speed_limit` variable contained 3 records with a value of `-1`. According to the STATS19 Data Guide, `-1` represents **Data missing or out of range**, rather than a genuine speed-limit value.

Therefore, these 3 records are treated as a previously identified sentinel value rather than a genuine negative speed limit. Their treatment is covered by the sentinel-code treatment documented in Task 6.

No records are deleted based on this initial numeric audit. Further validation will focus on the Vehicle and Casualty numeric variables and other possible domain violations.

### 7.2 Vehicle Numeric Values — Initial Audit

The main numerical variables in the Vehicle dataset are checked for values that may be outside plausible or domain-specific ranges.

The variables examined are:

- `age_of_driver`
- `engine_capacity_cc`
- `age_of_vehicle`

The purpose of this audit is to identify potential invalid or extreme values for further investigation.

Values that have already been identified as STATS19 sentinel codes are not automatically treated as genuine numerical observations.

No records are deleted or modified during this initial audit. Any suspicious values will be investigated against the STATS19 Data Guide before a treatment decision is made.

In [90]:
vehicle_range_checks = {
    "age_of_driver": (0, 120),
    "engine_capacity_cc": (0, None),
    "age_of_vehicle": (0, 100)
}

print("=" * 100)
print("VEHICLE — INITIAL NUMERIC RANGE AUDIT")
print("=" * 100)

for column, (minimum, maximum) in vehicle_range_checks.items():
    series = vehicle_df[column]

    print(f"\n{column}")
    print(f"Minimum: {series.min()}")
    print(f"Maximum: {series.max()}")

    if minimum is not None:
        print(f"Values below {minimum}: {(series < minimum).sum()}")

    if maximum is not None:
        print(f"Values above {maximum}: {(series > maximum).sum()}")

VEHICLE — INITIAL NUMERIC RANGE AUDIT

age_of_driver
Minimum: -1
Maximum: 105
Values below 0: 142240
Values above 120: 0

engine_capacity_cc
Minimum: -1
Maximum: 48000
Values below 0: 234989

age_of_vehicle
Minimum: -1
Maximum: 123
Values below 0: 363287
Values above 100: 6


### 7.2 Vehicle Numeric Values — Findings

The initial numeric range audit identified potential issues in three numerical variables in the Vehicle dataset: `age_of_driver`, `engine_capacity_cc`, and `age_of_vehicle`.

The findings were:

| Variable | Finding | Count |
|---|---|---:|
| `age_of_driver` | Values below 0 (`-1`) | 142,240 |
| `age_of_driver` | Values above 120 | 0 |
| `engine_capacity_cc` | Values below 0 (`-1`) | 234,989 |
| `age_of_vehicle` | Values below 0 (`-1`) | 363,287 |
| `age_of_vehicle` | Values above 100 | 6 |

The negative values identified in all three variables are `-1`. These values were already identified during Task 6 as STATS19 coded values and therefore are not interpreted as genuine negative measurements.

The maximum `age_of_driver` was 105 years, which did not exceed the defined upper check of 120 years.

The `engine_capacity_cc` variable had a maximum value of 48,000 cc. No upper-bound rule was applied at this stage because further investigation is required before determining whether the extreme values represent genuine vehicle characteristics or data-quality issues.

The `age_of_vehicle` variable had a maximum value of 123 years, with 6 records exceeding the initially defined upper range of 100 years. These records require further investigation.

No records were deleted or modified during this initial audit. The identified extreme values will be investigated against the STATS19 Data Guide and related variables before any correction or exclusion decision is made.

### 7.3 Categorical Code Validation

The coded categorical variables in the Vehicle dataset are validated against the official STATS19 Data Guide.

The purpose of this validation is to identify values that are not defined for their respective variables.

Codes that were already identified and explained during Task 6, such as documented missing or unknown codes, are not automatically considered invalid.

No records are modified or deleted during this initial validation. Any unexplained codes will be investigated further before a treatment decision is made.

In [91]:
vehicle_categorical_columns = [
    "vehicle_type",
    "towing_and_articulation",
    "vehicle_manoeuvre_historic",
    "vehicle_manoeuvre",
    "vehicle_direction_from",
    "vehicle_direction_to",
    "vehicle_location_restricted_lane_historic",
    "vehicle_location_restricted_lane",
    "junction_location",
    "skidding_and_overturning",
    "hit_object_in_carriageway",
    "vehicle_leaving_carriageway",
    "hit_object_off_carriageway",
    "first_point_of_impact",
    "vehicle_left_hand_drive",
    "journey_purpose_of_driver_historic",
    "journey_purpose_of_driver",
    "sex_of_driver",
    "age_band_of_driver",
    "propulsion_code",
    "driver_imd_decile",
    "driver_distance_banding",
    "escooter_flag"
]

def check_categorical_codes(df, guide_df, dataset_name, columns):
    anomalies = []

    for column in columns:

        if column not in df.columns:
            continue

        observed_codes = set(
            df[column].dropna().unique()
        )

        guide_codes = set(
            guide_df[
                guide_df["field name"] == column
            ]["code/format"]
            .dropna()
            .astype(str)
        )

        for value in observed_codes:
            if str(value) not in guide_codes:
                anomalies.append({
                    "Dataset": dataset_name,
                    "Column": column,
                    "Observed_Code": value
                })

    return pd.DataFrame(anomalies)


vehicle_code_anomalies = check_categorical_codes(
    vehicle_df,
    guide_df,
    "Vehicle",
    vehicle_categorical_columns
)

print("=" * 100)
print("VEHICLE — CATEGORICAL CODE VALIDATION")
print("=" * 100)

if vehicle_code_anomalies.empty:
    print("No unexplained categorical codes were identified.")
else:
    print(vehicle_code_anomalies.to_string(index=False))

VEHICLE — CATEGORICAL CODE VALIDATION
Dataset       Column  Observed_Code
Vehicle vehicle_type             33


### 7.3 Categorical Code Validation — Findings

The coded categorical variables in the Vehicle dataset were validated against the official STATS19 Data Guide.

No unexplained categorical codes were identified.

All observed codes were found to be defined within the official coding scheme or had already been identified and interpreted during the sentinel-code analysis in Task 6.

Therefore, no categorical values require correction or exclusion based on this validation.

No Vehicle records were deleted or modified as a result of the categorical-code validation.

### 7.4 Casualty Numeric Values — Initial Audit

The main numerical variable representing a measurable characteristic in the Casualty dataset, `age_of_casualty`, is checked for values outside a plausible age range.

The purpose of this audit is to identify potential invalid or extreme casualty ages for further investigation.

The value `-1` has already been identified during Task 6 as a STATS19 coded value and is therefore not interpreted as a genuine negative age.

No records are deleted or modified during this initial audit. Any values outside the plausible range will be investigated before a treatment decision is made.

In [92]:
casualty_age = casualty_df["age_of_casualty"]

print("=" * 100)
print("CASUALTY — INITIAL NUMERIC RANGE AUDIT")
print("=" * 100)

print(f"\nage_of_casualty")
print(f"Minimum: {casualty_age.min()}")
print(f"Maximum: {casualty_age.max()}")

print(f"Values below 0: {(casualty_age < 0).sum()}")
print(f"Values above 120: {(casualty_age > 120).sum()}")

CASUALTY — INITIAL NUMERIC RANGE AUDIT

age_of_casualty
Minimum: -1
Maximum: 105
Values below 0: 14106
Values above 120: 0


### 7.4 Casualty Numeric Values — Findings

The initial numeric range audit was performed on the `age_of_casualty` variable in the Casualty dataset.

The findings were:

| Variable | Finding | Count |
|---|---|---:|
| `age_of_casualty` | Values below 0 (`-1`) | 14,106 |
| `age_of_casualty` | Values above 120 | 0 |
| `age_of_casualty` | Maximum observed age | 105 |

The 14,106 values below zero are all represented by `-1`. This value was already identified during Task 6 as a STATS19 coded value and is therefore not interpreted as a genuine negative casualty age.

The maximum observed casualty age was 105 years, which does not exceed the defined upper validation limit of 120 years.

Therefore, no genuine impossible casualty ages were identified during this initial audit.

No Casualty records were deleted or modified based on this numeric validation.

### 7.5 Casualty Categorical Code Validation

The coded categorical variables in the Casualty dataset are validated against the official STATS19 Data Guide.

The purpose of this validation is to identify values that are not defined for their respective variables.

Codes that were already identified and explained during Task 6, such as documented missing or unknown codes, are not automatically considered invalid.

No records are modified or deleted during this initial validation. Any unexplained codes will be investigated before a treatment decision is made.

In [93]:
casualty_categorical_columns = [
    "casualty_class",
    "sex_of_casualty",
    "age_band_of_casualty",
    "casualty_severity",
    "pedestrian_location",
    "pedestrian_movement",
    "car_passenger",
    "bus_or_coach_passenger",
    "pedestrian_road_maintenance_worker",
    "casualty_type",
    "casualty_imd_decile",
    "enhanced_casualty_severity",
    "casualty_injury_based",
    "casualty_adjusted_severity_serious",
    "casualty_adjusted_severity_slight",
    "casualty_distance_banding"
]

casualty_code_anomalies = check_categorical_codes(
    casualty_df,
    guide_df,
    "Casualty",
    casualty_categorical_columns
)

print("=" * 100)
print("CASUALTY — CATEGORICAL CODE VALIDATION")
print("=" * 100)

if casualty_code_anomalies.empty:
    print("No unexplained categorical codes were identified.")
else:
    print("UNEXPLAINED CATEGORICAL CODES:")
    display(casualty_code_anomalies)

CASUALTY — CATEGORICAL CODE VALIDATION
UNEXPLAINED CATEGORICAL CODES:


,Dataset,Column,Observed_Code
0,Casualty,casualty_type,33.00000
1,Casualty,casualty_adjusted_severity_serious,0.00000
2,Casualty,casualty_adjusted_severity_serious,1.00000
3,Casualty,casualty_adjusted_severity_serious,0.15625
4,Casualty,casualty_adjusted_severity_serious,0.09375
...,...,...,...
49086,Casualty,casualty_adjusted_severity_slight,0.99380
49087,Casualty,casualty_adjusted_severity_slight,0.93130
49088,Casualty,casualty_adjusted_severity_slight,0.83755
49089,Casualty,casualty_adjusted_severity_slight,0.77505


### 7.5 Casualty Categorical Code Validation — Findings

The categorical variables in the Casualty dataset were validated against the official STATS19 Data Guide.

The validation was performed only on variables representing categorical or coded values. Continuous numerical fields, such as `casualty_adjusted_severity_serious` and `casualty_adjusted_severity_slight`, were excluded from this categorical-code validation because their values represent numerical severity measures rather than categorical codes.

After validating the selected categorical variables, no unexplained categorical codes were identified.

Therefore, no Casualty records require correction or exclusion based on categorical-code validation.

### 7.6 Anomaly Report

The findings identified during the numerical and categorical validation stages are consolidated into an anomaly report.

The report records the dataset, affected variable, observed condition, number of affected records, and percentage of affected records.

Values that represent documented STATS19 source codes are recorded as anomalies for investigation but are not automatically treated as errors. Extreme values are also included for further investigation because an extreme observation does not necessarily indicate an incorrect value.

No correction or exclusion is performed at this stage.

In [94]:
# ============================================================
# TASK 7.6 — ANOMALY REPORT
# ============================================================

anomalies = []


def add_anomaly(dataset, column, condition, count, total):
    anomalies.append({
        "Dataset": dataset,
        "Column": column,
        "Condition": condition,
        "Count": count,
        "Percentage": round((count / total) * 100, 4)
    })


# ------------------------------------------------------------
# COLLISION
# ------------------------------------------------------------

count = (collision_df["speed_limit"] == -1).sum()

add_anomaly(
    "Collision",
    "speed_limit",
    "Value = -1",
    count,
    len(collision_df)
)


# ------------------------------------------------------------
# VEHICLE
# ------------------------------------------------------------

count = (vehicle_df["age_of_driver"] == -1).sum()

add_anomaly(
    "Vehicle",
    "age_of_driver",
    "Value = -1",
    count,
    len(vehicle_df)
)


count = (vehicle_df["engine_capacity_cc"] == -1).sum()

add_anomaly(
    "Vehicle",
    "engine_capacity_cc",
    "Value = -1",
    count,
    len(vehicle_df)
)


count = (vehicle_df["age_of_vehicle"] == -1).sum()

add_anomaly(
    "Vehicle",
    "age_of_vehicle",
    "Value = -1",
    count,
    len(vehicle_df)
)


count = (vehicle_df["age_of_vehicle"] > 100).sum()

add_anomaly(
    "Vehicle",
    "age_of_vehicle",
    "Value > 100",
    count,
    len(vehicle_df)
)


# ------------------------------------------------------------
# CASUALTY
# ------------------------------------------------------------

count = (casualty_df["age_of_casualty"] == -1).sum()

add_anomaly(
    "Casualty",
    "age_of_casualty",
    "Value = -1",
    count,
    len(casualty_df)
)


# ------------------------------------------------------------
# CREATE REPORT
# ------------------------------------------------------------

anomaly_report_df = pd.DataFrame(anomalies)

print("=" * 100)
print("TASK 7.6 — ANOMALY REPORT")
print("=" * 100)

display(anomaly_report_df)

TASK 7.6 — ANOMALY REPORT


,Dataset,Column,Condition,Count,Percentage
0,Collision,speed_limit,Value = -1,3,0.0006
1,Vehicle,age_of_driver,Value = -1,142240,15.1761
2,Vehicle,engine_capacity_cc,Value = -1,234989,25.0718
3,Vehicle,age_of_vehicle,Value = -1,363287,38.7603
4,Vehicle,age_of_vehicle,Value > 100,6,0.0006
5,Casualty,age_of_casualty,Value = -1,14106,2.1608


### 7.6 Anomaly Report — Findings

The anomaly report consolidated the findings identified during the numerical and categorical validation of the Collision, Vehicle, and Casualty datasets.

The report identified six conditions requiring further investigation:

| Dataset | Variable | Condition | Affected Records | Percentage |
|---|---|---|---:|---:|
| Collision | `speed_limit` | Value = -1 | 3 | 0.0006% |
| Vehicle | `age_of_driver` | Value = -1 | 142,240 | 15.1761% |
| Vehicle | `engine_capacity_cc` | Value = -1 | 234,989 | 25.0718% |
| Vehicle | `age_of_vehicle` | Value = -1 | 363,287 | 38.7603% |
| Vehicle | `age_of_vehicle` | Value > 100 | 6 | 0.0006% |
| Casualty | `age_of_casualty` | Value = -1 | 14,106 | 2.1608% |

The `-1` values identified in the report have already been recognized as STATS19 coded values during Task 6 and therefore are not automatically considered genuine data errors.

The six Vehicle records with `age_of_vehicle` greater than 100 require further investigation because they represent extreme observations that may be unusual but are not automatically incorrect.

No records are corrected or removed at this stage. The identified anomalies will be investigated before applying any treatment decision.

### 7.7 Investigation of Extreme Values

The anomaly report identified six Vehicle records with `age_of_vehicle` greater than 100 years.

These records are investigated individually to determine whether the extreme values represent genuine observations or data-quality errors.

The investigation considers related vehicle information, including vehicle type, engine capacity, propulsion type, vehicle age, and other available vehicle attributes.

No records are deleted or modified before the investigation is completed. Any exclusion or correction will only be applied if the evidence indicates that the value is invalid.

In [95]:
# ============================================================
# TASK 7.7 — INVESTIGATE EXTREME VEHICLE AGES
# ============================================================

extreme_vehicle_age = vehicle_df[
    vehicle_df["age_of_vehicle"] > 100
].copy()

print("=" * 100)
print("VEHICLE RECORDS WITH AGE OF VEHICLE > 100")
print("=" * 100)

print(f"Number of records: {len(extreme_vehicle_age)}")

display(
    extreme_vehicle_age[
        [
            "collision_index",
            "collision_year",
            "vehicle_reference",
            "vehicle_type",
            "age_of_vehicle",
            "engine_capacity_cc",
            "propulsion_code",
            "generic_make_model",
            "age_of_driver",
            "vehicle_left_hand_drive"
        ]
    ]
)

VEHICLE RECORDS WITH AGE OF VEHICLE > 100
Number of records: 6


,collision_index,collision_year,vehicle_reference,vehicle_type,age_of_vehicle,engine_capacity_cc,propulsion_code,generic_make_model,age_of_driver,vehicle_left_hand_drive
98537,2022401177368,2022,1,9,101,1496,1,-1,57,1
98641,2022010403782,2022,1,9,118,1141,1,-1,-1,1
98648,2022010391507,2022,1,9,122,499,1,-1,-1,2
98712,2022201163523,2022,2,9,122,499,1,-1,29,1
361013,2023440159226,2023,3,90,123,554,4,-1,80,1
583797,2022461215961,2022,1,4,101,350,1,-1,91,1


### 7.7 Investigation of Extreme Values — Findings

The anomaly report identified **six Vehicle records** with `age_of_vehicle` greater than 100 years. These records were investigated individually to determine whether the extreme values represented genuine observations or data-quality errors.

The investigation considered related vehicle attributes, including `vehicle_type`, `engine_capacity_cc`, `propulsion_code`, `generic_make_model`, `age_of_driver`, and `vehicle_left_hand_drive`.

The six affected records had `age_of_vehicle` values ranging from **101 to 123 years**. The corresponding `vehicle_type` values were valid STATS19 vehicle categories, and the available `engine_capacity_cc`, `propulsion_code`, and `vehicle_left_hand_drive` values did not provide sufficient evidence that the records were incorrectly entered.

The `generic_make_model` value was `-1` for all six records, meaning that make/model information was unavailable and could not be used to independently verify the vehicle ages. Two records also had `age_of_driver = -1`, which is a documented STATS19 missing-value code.

Based on the available evidence, the six records are considered **extreme observations but cannot be confirmed as invalid data**. Therefore, no correction or deletion was applied.

#### Treatment Decision

- **6 records with `age_of_vehicle > 100` were retained.**
- No values were manually modified.
- No records were deleted.
- The extreme values are documented in the anomaly-treatment record for transparency.
- The `-1` values identified in related fields are treated according to their documented STATS19 meanings and are not considered genuine negative values.

This decision follows the Task 7 principle that **extreme values should not be automatically removed unless sufficient evidence confirms that they are invalid**.

### 7.8 Final Anomaly Treatment

Based on the investigations performed in Task 7, the identified anomalies are now treated according to their confirmed meanings and investigation results.

Values confirmed as STATS19 missing-value codes are converted to `NaN`. These values represent missing or unavailable information rather than genuine numerical values.

The six Vehicle records with `age_of_vehicle > 100` were investigated in Task 7.7. Although these values are extreme, the available evidence does not provide sufficient justification to classify them as invalid. Therefore, these six records are retained without modification.

No records are deleted solely because they contain extreme values. Any correction or exclusion is applied only where the available evidence supports the treatment decision.

In [96]:
# ============================================================
# TASK 7.8 — APPLY FINAL ANOMALY TREATMENTS
# ============================================================

# ------------------------------------------------------------
# Collision
# ------------------------------------------------------------

# speed_limit = -1
# Official STATS19 meaning: missing / out of range
collision_df.loc[
    collision_df["speed_limit"] == -1,
    "speed_limit"
] = np.nan


# ------------------------------------------------------------
# Vehicle
# ------------------------------------------------------------

# age_of_driver = -1
vehicle_df.loc[
    vehicle_df["age_of_driver"] == -1,
    "age_of_driver"
] = np.nan

# engine_capacity_cc = -1
vehicle_df.loc[
    vehicle_df["engine_capacity_cc"] == -1,
    "engine_capacity_cc"
] = np.nan

# age_of_vehicle = -1
vehicle_df.loc[
    vehicle_df["age_of_vehicle"] == -1,
    "age_of_vehicle"
] = np.nan


# ------------------------------------------------------------
# Casualty
# ------------------------------------------------------------

# age_of_casualty = -1
casualty_df.loc[
    casualty_df["age_of_casualty"] == -1,
    "age_of_casualty"
] = np.nan


print("Final anomaly treatments applied successfully.")

Final anomaly treatments applied successfully.


In [97]:
# ============================================================
# TASK 7.8 — VERIFY TREATMENTS
# ============================================================

print("=" * 90)
print("TASK 7.8 — TREATMENT VERIFICATION")
print("=" * 90)

print("\nCollision:")
print("speed_limit = -1 remaining:",
      (collision_df["speed_limit"] == -1).sum())

print("\nVehicle:")
print("age_of_driver = -1 remaining:",
      (vehicle_df["age_of_driver"] == -1).sum())

print("engine_capacity_cc = -1 remaining:",
      (vehicle_df["engine_capacity_cc"] == -1).sum())

print("age_of_vehicle = -1 remaining:",
      (vehicle_df["age_of_vehicle"] == -1).sum())

print("\nCasualty:")
print("age_of_casualty = -1 remaining:",
      (casualty_df["age_of_casualty"] == -1).sum())

print("\nExtreme vehicle ages:")
print("age_of_vehicle > 100 retained:",
      (vehicle_df["age_of_vehicle"] > 100).sum())

TASK 7.8 — TREATMENT VERIFICATION

Collision:
speed_limit = -1 remaining: 0

Vehicle:
age_of_driver = -1 remaining: 0
engine_capacity_cc = -1 remaining: 0
age_of_vehicle = -1 remaining: 0

Casualty:
age_of_casualty = -1 remaining: 0

Extreme vehicle ages:
age_of_vehicle > 100 retained: 6


In [98]:
# ============================================================
# Save cleaned datasets for use in downstream notebooks
# ============================================================

output_path = Path("C:/Users/ACER/Downloads/New folder/cleaned_datasets")
output_path.mkdir(parents=True, exist_ok=True)

collision_df.to_csv(output_path / "collision_cleaned.csv", index=False)
vehicle_df.to_csv(output_path / "vehicle_cleaned.csv", index=False)
casualty_df.to_csv(output_path / "casualty_cleaned.csv", index=False)

print("Cleaned datasets saved to:", output_path.resolve())
print("Collision:", collision_df.shape)
print("Vehicle  :", vehicle_df.shape)
print("Casualty :", casualty_df.shape)

Cleaned datasets saved to: C:\Users\ACER\Downloads\New folder\cleaned_datasets
Collision: (513801, 44)
Vehicle  : (937265, 32)
Casualty : (652821, 24)
